# 05 — Ordinal Models

This notebook repeats the Notebook 04 benchmark while explicitly using:

\[
\mathrm{No} < \mathrm{Less} < \mathrm{Most}
\]

The compounds, feature representations, outer folds, inner grouping rules, tuning objective, and evaluation metrics are kept matched to Notebook 04.

## Ordinal strategy

For the ordered target encoded as:

- No = 0
- Less = 1
- Most = 2

we fit two cumulative binary models:

1. `y > 0`
2. `y > 1`

The final prediction is the number of thresholds crossed.

This is a standard ordinal reduction approach and lets us compare the same underlying learner families in nominal and ordinal form.


## 1. Imports and paths

In [2]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, recall_score,
    confusion_matrix, mean_absolute_error, cohen_kappa_score,
)

RANDOM_STATE = 42
INNER_SPLITS = 3

candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (p for p in candidate_roots if (p / "data/features/feature_ids.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Notebook 03 outputs not found.")

FEATURE_DIR = PROJECT_ROOT / "data/features"
SPLIT_DIR = PROJECT_ROOT / "data/splits"
TABLE_DIR = PROJECT_ROOT / "results/tables"
PRED_DIR = PROJECT_ROOT / "results/predictions"
FIG_DIR = PROJECT_ROOT / "results/figures/ordinal"

for p in [TABLE_DIR, PRED_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Project root: /home/gman/Documents/BS-Bioinformatics/dilirank2-prediction


## 2. Load the same features and outer folds as Notebook 04

In [3]:
feature_ids = pd.read_csv(FEATURE_DIR / "feature_ids.csv")
descriptor_file = pd.read_csv(FEATURE_DIR / "rdkit_descriptors.csv")
population = pd.read_csv(SPLIT_DIR / "concern_modelling_population.csv")
morgan_npz = np.load(FEATURE_DIR / "morgan_ecfp4_2048.npz", allow_pickle=True)
X_morgan_all = morgan_npz["X"]

assert np.array_equal(feature_ids["feature_row"].to_numpy(), np.arange(len(feature_ids)))

descriptor_columns = [c for c in descriptor_file.columns if c not in {"feature_row", "LTKBID"}]
X_descriptor_df = (
    descriptor_file.sort_values("feature_row")[descriptor_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

FLOAT32_LIMIT = np.finfo(np.float32).max
too_large = X_descriptor_df.abs() > FLOAT32_LIMIT
print("RDKit values outside float32 range converted to NaN:", int(too_large.to_numpy().sum()))
X_descriptor_df = X_descriptor_df.mask(too_large, np.nan)

model_rows = population["feature_row"].to_numpy(dtype=int)
X_descriptor = X_descriptor_df.iloc[model_rows].to_numpy(dtype=np.float64)
X_morgan = X_morgan_all[model_rows].astype(np.uint8)

y_order = population["DILI_order"].astype(int).to_numpy()
y_class = population["DILI_class"].astype(str).to_numpy()

CLASS_ORDER = ["No", "Less", "Most"]
ORDER_TO_CLASS = {0: "No", 1: "Less", 2: "Most"}

print("RDKit:", X_descriptor.shape, "| Morgan:", X_morgan.shape)
display(population["DILI_class"].value_counts().reindex(CLASS_ORDER).to_frame("count"))


RDKit values outside float32 range converted to NaN: 20
RDKit: (886, 217) | Morgan: (886, 2048)


,count
DILI_class,
No,350
Less,330
Most,206


## 3. Cumulative ordinal classifier

In [4]:
class OrdinalThresholdClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        y = np.asarray(y, dtype=int)
        self.classes_ = np.sort(np.unique(y))

        if not np.array_equal(self.classes_, np.arange(len(self.classes_))):
            raise ValueError("Ordinal classes must be consecutive integers starting at 0.")

        self.thresholds_ = self.classes_[:-1]
        self.models_ = []

        for threshold in self.thresholds_:
            binary_y = (y > threshold).astype(int)
            model = clone(self.base_estimator)
            model.fit(X, binary_y)
            self.models_.append(model)

        return self

    def predict(self, X):
        binary_preds = np.column_stack([
            np.asarray(model.predict(X), dtype=int)
            for model in self.models_
        ])

        # Enforce cumulative consistency.
        monotone = binary_preds.copy()
        for j in range(1, monotone.shape[1]):
            monotone[:, j] = np.minimum(
                monotone[:, j],
                monotone[:, j - 1],
            )

        return monotone.sum(axis=1).astype(int)


## 4. Leakage-safe preprocessing

In [5]:
def make_pipeline(representation, model_name, ordinal_classifier):
    steps = []

    if representation == "rdkit":
        steps += [
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("variance", VarianceThreshold(0.0)),
        ]
        if model_name in {"ordinal_logistic", "ordinal_svm"}:
            steps.append(("scaler", StandardScaler()))

    elif representation == "morgan_ecfp4":
        steps.append(("variance", VarianceThreshold(0.0)))

    else:
        raise ValueError(representation)

    steps.append(("clf", ordinal_classifier))
    return Pipeline(steps)


## 5. Ordinal base learners and compact tuning grids

In [6]:
ORDINAL_MODEL_SPECS = {
    "ordinal_logistic": {
        "base_estimator": LogisticRegression(
            class_weight="balanced", max_iter=5000, solver="lbfgs",
            random_state=RANDOM_STATE
        ),
        "param_grid": {"clf__base_estimator__C": [0.1, 1.0, 10.0]},
    },

    "ordinal_svm": {
        "base_estimator": SVC(
            kernel="rbf", class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "param_grid": {
            "clf__base_estimator__C": [1.0, 4.0],
            "clf__base_estimator__gamma": ["scale", "auto"],
        },
    },

    "ordinal_random_forest": {
        "base_estimator": RandomForestClassifier(
            n_estimators=300, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE
        ),
        "param_grid": {
            "clf__base_estimator__max_depth": [None, 12],
            "clf__base_estimator__min_samples_leaf": [1, 3],
            "clf__base_estimator__max_features": ["sqrt"],
        },
    },

    "ordinal_hist_gradient_boosting": {
        "base_estimator": HistGradientBoostingClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "param_grid": {
            "clf__base_estimator__learning_rate": [0.05, 0.10],
            "clf__base_estimator__max_leaf_nodes": [15, 31],
            "clf__base_estimator__l2_regularization": [0.0],
        },
    },
}


## 6. Metrics and grouped inner CV

In [7]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    true_class = np.array([ORDER_TO_CLASS[x] for x in y_true])
    pred_class = np.array([ORDER_TO_CLASS[x] for x in y_pred])

    recalls = recall_score(
        true_class, pred_class,
        labels=CLASS_ORDER, average=None, zero_division=0
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "recall_No": recalls[0],
        "recall_Less": recalls[1],
        "recall_Most": recalls[2],
        "ordinal_mae": mean_absolute_error(y_true, y_pred),
        "quadratic_weighted_kappa": cohen_kappa_score(
            y_true, y_pred, weights="quadratic"
        ),
    }

def make_inner_cv():
    return StratifiedGroupKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )


## 7. Quick ordinal sanity check

In [8]:
outer = population["random_fold"].astype(int).to_numpy()
test_mask = outer == 0
train_mask = ~test_mask

rows = []
for model_name, spec in ORDINAL_MODEL_SPECS.items():
    clf = OrdinalThresholdClassifier(
        base_estimator=clone(spec["base_estimator"])
    )

    pipe = make_pipeline("rdkit", model_name, clf)
    pipe.fit(X_descriptor[train_mask], y_order[train_mask])

    pred = pipe.predict(X_descriptor[test_mask])
    ba = balanced_accuracy_score(y_order[test_mask], pred)

    rows.append({"model": model_name, "balanced_accuracy": ba, "status": "OK"})
    print(model_name, "OK", f"BA={ba:.3f}")

display(pd.DataFrame(rows))


ordinal_logistic OK BA=0.464
ordinal_svm OK BA=0.462
ordinal_random_forest OK BA=0.482
ordinal_hist_gradient_boosting OK BA=0.411


,model,balanced_accuracy,status
0,ordinal_logistic,0.463781,OK
1,ordinal_svm,0.462482,OK
2,ordinal_random_forest,0.481962,OK
3,ordinal_hist_gradient_boosting,0.410967,OK


## 8. Nested ordinal benchmark

In [9]:
REPRESENTATIONS = {
    "rdkit": X_descriptor,
    "morgan_ecfp4": X_morgan,
}
SPLIT_REGIMES = {
    "random": {"fold_column": "random_fold", "group_column": "parent_group"},
    "scaffold": {"fold_column": "scaffold_fold", "group_column": "scaffold_group"},
}

fold_result_rows = []
prediction_rows = []

checkpoint_folds = TABLE_DIR / "ordinal_fold_results_checkpoint.csv"
checkpoint_preds = PRED_DIR / "ordinal_predictions_checkpoint.csv"

for representation_name, X in REPRESENTATIONS.items():
    for split_name, info in SPLIT_REGIMES.items():

        outer_folds = population[info["fold_column"]].astype(int).to_numpy()
        groups_all = population[info["group_column"]].astype(str).to_numpy()

        for model_name, spec in ORDINAL_MODEL_SPECS.items():
            print("\n" + "=" * 80)
            print(representation_name, split_name, model_name)

            for outer_fold in sorted(np.unique(outer_folds)):
                print("Outer fold", outer_fold)

                test_mask = outer_folds == outer_fold
                train_mask = ~test_mask

                ordinal_clf = OrdinalThresholdClassifier(
                    base_estimator=clone(spec["base_estimator"])
                )

                pipeline = make_pipeline(
                    representation_name,
                    model_name,
                    ordinal_clf,
                )

                search = GridSearchCV(
                    pipeline,
                    spec["param_grid"],
                    scoring="balanced_accuracy",
                    cv=make_inner_cv(),
                    n_jobs=-1,
                    refit=True,
                    error_score="raise",
                    verbose=1,
                )

                start = time.perf_counter()
                search.fit(
                    X[train_mask],
                    y_order[train_mask],
                    groups=groups_all[train_mask],
                )
                elapsed = time.perf_counter() - start

                y_test = y_order[test_mask]
                y_pred = search.predict(X[test_mask])

                metrics = compute_metrics(y_test, y_pred)

                fold_result_rows.append({
                    "representation": representation_name,
                    "split_regime": split_name,
                    "model": model_name,
                    "outer_fold": int(outer_fold),
                    "n_train": int(train_mask.sum()),
                    "n_test": int(test_mask.sum()),
                    "inner_best_balanced_accuracy": float(search.best_score_),
                    "best_params": json.dumps(search.best_params_, sort_keys=True),
                    "fit_seconds": elapsed,
                    **metrics,
                })

                tp = population.loc[test_mask].reset_index(drop=True)

                for i in range(len(tp)):
                    prediction_rows.append({
                        "feature_row": int(tp.loc[i, "feature_row"]),
                        "LTKBID": tp.loc[i, "LTKBID"],
                        "CompoundName": tp.loc[i, "CompoundName"],
                        "true_class": ORDER_TO_CLASS[int(y_test[i])],
                        "true_order": int(y_test[i]),
                        "predicted_class": ORDER_TO_CLASS[int(y_pred[i])],
                        "predicted_order": int(y_pred[i]),
                        "representation": representation_name,
                        "split_regime": split_name,
                        "model": model_name,
                        "outer_fold": int(outer_fold),
                    })

                pd.DataFrame(fold_result_rows).to_csv(checkpoint_folds, index=False)
                pd.DataFrame(prediction_rows).to_csv(checkpoint_preds, index=False)

                print(
                    f"BA={metrics['balanced_accuracy']:.3f} | "
                    f"Macro-F1={metrics['macro_f1']:.3f} | "
                    f"MAE={metrics['ordinal_mae']:.3f} | "
                    f"QWK={metrics['quadratic_weighted_kappa']:.3f} | "
                    f"{elapsed:.1f}s"
                )



rdkit random ordinal_logistic
Outer fold 0
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.497 | Macro-F1=0.489 | MAE=0.618 | QWK=0.337 | 2.4s
Outer fold 1
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.504 | Macro-F1=0.489 | MAE=0.644 | QWK=0.298 | 1.9s
Outer fold 2
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.483 | Macro-F1=0.473 | MAE=0.667 | QWK=0.266 | 0.4s
Outer fold 3
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.469 | Macro-F1=0.457 | MAE=0.684 | QWK=0.268 | 0.4s
Outer fold 4
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.476 | Macro-F1=0.452 | MAE=0.678 | QWK=0.299 | 0.4s

rdkit random ordinal_svm
Outer fold 0
Fitting 3 folds for each of 4 candidates, totalling 12 fits
BA=0.462 | Macro-F1=0.456 | MAE=0.674 | QWK=0.264 | 0.4s
Outer fold 1
Fitting 3 folds for each of 4 candidates, totalling 12 fits
BA=0.487 | Macro-F1=0.473 | MAE=0.650 | QWK=0.311 | 0.5s
Outer fold 2
Fitting 3 folds for each 

## 9. Save and summarize

In [10]:
fold_results = pd.DataFrame(fold_result_rows)
predictions = pd.DataFrame(prediction_rows)

fold_results.to_csv(TABLE_DIR / "ordinal_fold_results.csv", index=False)
predictions.to_csv(PRED_DIR / "ordinal_predictions.csv", index=False)

summary = (
    fold_results
    .groupby(["representation", "split_regime", "model"], as_index=False)
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        ordinal_mae_mean=("ordinal_mae", "mean"),
        ordinal_mae_std=("ordinal_mae", "std"),
        qwk_mean=("quadratic_weighted_kappa", "mean"),
        qwk_std=("quadratic_weighted_kappa", "std"),
        recall_No_mean=("recall_No", "mean"),
        recall_Less_mean=("recall_Less", "mean"),
        recall_Most_mean=("recall_Most", "mean"),
        mean_fit_seconds=("fit_seconds", "mean"),
    )
    .sort_values(["split_regime", "balanced_accuracy_mean"], ascending=[True, False])
)

summary.to_csv(TABLE_DIR / "ordinal_summary.csv", index=False)
display(summary.round(3))


,representation,split_regime,model,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,accuracy_mean,ordinal_mae_mean,ordinal_mae_std,qwk_mean,qwk_std,recall_No_mean,recall_Less_mean,recall_Most_mean,mean_fit_seconds
3,morgan_ecfp4,random,ordinal_svm,0.512,0.047,0.514,0.045,0.525,0.547,0.042,0.395,0.039,0.549,0.561,0.427,13.532
1,morgan_ecfp4,random,ordinal_logistic,0.511,0.042,0.508,0.039,0.519,0.571,0.039,0.383,0.053,0.606,0.461,0.467,4.788
8,rdkit,random,ordinal_hist_gradient_boosting,0.506,0.051,0.509,0.053,0.526,0.542,0.064,0.389,0.057,0.569,0.579,0.369,67.728
0,morgan_ecfp4,random,ordinal_hist_gradient_boosting,0.504,0.047,0.502,0.047,0.516,0.578,0.061,0.368,0.058,0.614,0.461,0.437,59.004
2,morgan_ecfp4,random,ordinal_random_forest,0.504,0.048,0.506,0.047,0.518,0.551,0.054,0.393,0.068,0.549,0.555,0.408,13.536
9,rdkit,random,ordinal_logistic,0.486,0.015,0.472,0.018,0.483,0.658,0.027,0.294,0.029,0.600,0.333,0.524,1.073
11,rdkit,random,ordinal_svm,0.484,0.014,0.470,0.009,0.481,0.660,0.013,0.301,0.026,0.591,0.336,0.525,0.425
10,rdkit,random,ordinal_random_forest,0.460,0.028,0.457,0.027,0.495,0.550,0.041,0.346,0.054,0.497,0.670,0.214,6.664
5,morgan_ecfp4,scaffold,ordinal_logistic,0.511,0.020,0.509,0.019,0.524,0.579,0.040,0.359,0.059,0.626,0.469,0.437,3.890
4,morgan_ecfp4,scaffold,ordinal_hist_gradient_boosting,0.508,0.031,0.508,0.030,0.519,0.575,0.046,0.360,0.064,0.589,0.494,0.442,50.764


## 10. Ordinal error severity

In [11]:
predictions["absolute_error"] = (
    predictions["true_order"] - predictions["predicted_order"]
).abs()

predictions["error_type"] = np.select(
    [
        predictions["absolute_error"] == 0,
        predictions["absolute_error"] == 1,
        predictions["absolute_error"] == 2,
    ],
    ["correct", "adjacent", "extreme"],
    default="other",
)

error_summary = (
    predictions
    .groupby(["representation", "split_regime", "model", "error_type"])
    .size()
    .rename("count")
    .reset_index()
)

error_summary["percentage_within_model"] = (
    error_summary["count"]
    / error_summary.groupby(
        ["representation", "split_regime", "model"]
    )["count"].transform("sum")
    * 100
)

display(error_summary)


,representation,split_regime,model,error_type,count,percentage_within_model
0,morgan_ecfp4,random,ordinal_hist_gradient_boosting,adjacent,346,39.051919
1,morgan_ecfp4,random,ordinal_hist_gradient_boosting,correct,457,51.580135
2,morgan_ecfp4,random,ordinal_hist_gradient_boosting,extreme,83,9.367946
3,morgan_ecfp4,random,ordinal_logistic,adjacent,346,39.051919
4,morgan_ecfp4,random,ordinal_logistic,correct,460,51.918736
5,morgan_ecfp4,random,ordinal_logistic,extreme,80,9.029345
6,morgan_ecfp4,random,ordinal_random_forest,adjacent,366,41.309255
7,morgan_ecfp4,random,ordinal_random_forest,correct,459,51.805869
8,morgan_ecfp4,random,ordinal_random_forest,extreme,61,6.884876
9,morgan_ecfp4,random,ordinal_svm,adjacent,357,40.293454


## 11. Quick nominal-vs-ordinal preview

In [12]:
nominal_path = TABLE_DIR / "nominal_summary.csv"

if nominal_path.exists():
    nominal_summary = pd.read_csv(nominal_path)

    print("Nominal:")
    display(
        nominal_summary[
            [
                "representation", "split_regime", "model",
                "balanced_accuracy_mean", "macro_f1_mean",
                "ordinal_mae_mean", "qwk_mean",
            ]
        ].round(3)
    )

    print("Ordinal:")
    display(
        summary[
            [
                "representation", "split_regime", "model",
                "balanced_accuracy_mean", "macro_f1_mean",
                "ordinal_mae_mean", "qwk_mean",
            ]
        ].round(3)
    )
else:
    print("Run Notebook 04 first to create nominal_summary.csv.")


Nominal:


,representation,split_regime,model,balanced_accuracy_mean,macro_f1_mean,ordinal_mae_mean,qwk_mean
0,morgan_ecfp4,random,random_forest,0.537,0.529,0.554,0.421
1,morgan_ecfp4,random,svm,0.524,0.523,0.546,0.401
2,morgan_ecfp4,random,logistic_regression,0.517,0.513,0.570,0.387
3,rdkit,random,svm,0.505,0.498,0.621,0.321
4,rdkit,random,random_forest,0.501,0.502,0.571,0.349
5,rdkit,random,hist_gradient_boosting,0.498,0.498,0.580,0.345
6,morgan_ecfp4,random,hist_gradient_boosting,0.497,0.496,0.594,0.335
7,rdkit,random,logistic_regression,0.489,0.481,0.654,0.266
8,morgan_ecfp4,scaffold,random_forest,0.512,0.505,0.609,0.320
9,morgan_ecfp4,scaffold,logistic_regression,0.502,0.500,0.595,0.332


Ordinal:


,representation,split_regime,model,balanced_accuracy_mean,macro_f1_mean,ordinal_mae_mean,qwk_mean
3,morgan_ecfp4,random,ordinal_svm,0.512,0.514,0.547,0.395
1,morgan_ecfp4,random,ordinal_logistic,0.511,0.508,0.571,0.383
8,rdkit,random,ordinal_hist_gradient_boosting,0.506,0.509,0.542,0.389
0,morgan_ecfp4,random,ordinal_hist_gradient_boosting,0.504,0.502,0.578,0.368
2,morgan_ecfp4,random,ordinal_random_forest,0.504,0.506,0.551,0.393
9,rdkit,random,ordinal_logistic,0.486,0.472,0.658,0.294
11,rdkit,random,ordinal_svm,0.484,0.470,0.660,0.301
10,rdkit,random,ordinal_random_forest,0.460,0.457,0.550,0.346
5,morgan_ecfp4,scaffold,ordinal_logistic,0.511,0.509,0.579,0.359
4,morgan_ecfp4,scaffold,ordinal_hist_gradient_boosting,0.508,0.508,0.575,0.360


## Notebook Complete

Notebook 05 is the matched ordinal counterpart to Notebook 04.

Notebook 06 should perform the final comparison of:

- nominal vs ordinal;
- RDKit vs Morgan;
- random vs scaffold;
- balanced accuracy / macro-F1;
- ordinal MAE / quadratic weighted kappa;
- adjacent vs extreme errors.
